# CSVW-SAFE DEMO

Install the library with pip

In [1]:
#!pip install csvw-safe

In [2]:
import json
import pandas as pd
from csvw_safe import assert_same_structure, make_dummy_from_metadata, make_metadata_from_data, validate_metadata

In [3]:
import warnings  # pandas deprecation warning to update in library
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_csv("penguin_plus.csv", parse_dates=["timestamp", "timestamp_with_time"])
df.head(2)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Adelie,Torgersen,39.1,18.7,NaN,3750.0,True,7,2025-04-13,2025-04-13 15:04:00,1
1,Adelie,Torgersen,39.5,17.4,NaN,3800.0,False,1,2025-12-15,2025-12-15 18:15:26,2


## Table Level DP

In [7]:
metadata = make_metadata_from_data(df, privacy_unit="penguin_id", with_dependencies=False)
with open("metadata/penguin_metadata_table_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [8]:
# (Strict) Pydantic model
validated_metadata = validate_metadata(metadata)

In [9]:
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(5)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,f,g,NaN,17.271202,<NA>,5949,False,65,2025-03-20,2025-12-19 06:47:46,0
1,f,b,42.977825,15.568964,229,4120,True,340,2025-05-14,2025-02-12 06:47:46,4
2,f,d,54.151716,18.749224,211,2944,True,269,2025-03-10,2025-07-22 06:47:46,2
3,j,j,51.422932,16.650079,213,3970,False,137,2025-04-29,2025-06-02 06:47:46,2
4,i,j,54.439223,21.347358,<NA>,4380,True,133,2025-07-06,2025-02-18 06:47:46,2


In [10]:
# Should pass
assert_same_structure(df, dummy_df, check_categories=False)

In [11]:
# Should error as the metadata did not contain categories
assert_same_structure(df, dummy_df, check_categories=True)

AssertionError: Column 'species' dummy values {'i', 'g', 'j', 'c', 'b', 'h', 'f', 'd', 'e', 'a'} are not subset of original {'Gentoo', 'Adelie', 'Chinstrap'}

In [12]:
# Any combination
dummy_df.groupby(["species", "island"]).groups.keys()

dict_keys([('a', 'a'), ('a', 'b'), ('a', 'd'), ('a', 'e'), ('a', 'f'), ('a', 'g'), ('a', 'h'), ('a', 'i'), ('b', 'f'), ('b', 'g'), ('b', 'j'), ('c', 'b'), ('c', 'c'), ('c', 'd'), ('c', 'f'), ('c', 'g'), ('d', 'c'), ('d', 'd'), ('d', 'e'), ('d', 'g'), ('d', 'h'), ('d', 'i'), ('d', 'j'), ('e', 'b'), ('e', 'f'), ('e', 'g'), ('e', 'h'), ('e', 'j'), ('f', 'a'), ('f', 'b'), ('f', 'c'), ('f', 'd'), ('f', 'e'), ('f', 'g'), ('f', 'h'), ('f', 'i'), ('f', 'j'), ('g', 'e'), ('g', 'f'), ('g', 'h'), ('g', 'j'), ('h', 'a'), ('h', 'c'), ('h', 'e'), ('h', 'f'), ('h', 'h'), ('h', 'i'), ('h', 'j'), ('i', 'a'), ('i', 'b'), ('i', 'd'), ('i', 'e'), ('i', 'i'), ('i', 'j'), ('j', 'a'), ('j', 'b'), ('j', 'g'), ('j', 'i'), ('j', 'j')])

In [13]:
# No ordering (timestamp_with_time after timestamp)
assert (dummy_df["timestamp_with_time"] > dummy_df["timestamp"]).all(), "timestamp_with_time is not always greater than timestamp"

AssertionError: timestamp_with_time is not always greater than timestamp

## Table Level DP with Keys

In [14]:
metadata = make_metadata_from_data(
    df, privacy_unit="penguin_id", default_contributions_level="table_with_keys", with_dependencies=False
)
with open("metadata/penguin_metadata_table_with_keys_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [15]:
validated_metadata = validate_metadata(metadata)

In [16]:
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(5)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Chinstrap,Dream,NaN,17.271202,<NA>,5949,False,65,2025-03-20,2025-12-19 06:47:46,0
1,Chinstrap,Biscoe,42.977825,15.568964,229,4120,True,340,2025-05-14,2025-02-12 06:47:46,4
2,Chinstrap,Biscoe,54.151716,18.749224,211,2944,True,269,2025-03-10,2025-07-22 06:47:46,2
3,Gentoo,Torgersen,51.422932,16.650079,213,3970,False,137,2025-04-29,2025-06-02 06:47:46,2
4,Gentoo,Torgersen,54.439223,21.347358,<NA>,4380,True,133,2025-07-06,2025-02-18 06:47:46,2


In [17]:
# Should pass
assert_same_structure(df, dummy_df, check_categories=False)
assert_same_structure(df, dummy_df, check_categories=True)

In [18]:
# Any combination
dummy_df.groupby(["species", "island"]).groups.keys()

dict_keys([('Adelie', 'Biscoe'), ('Adelie', 'Dream'), ('Adelie', 'Torgersen'), ('Chinstrap', 'Biscoe'), ('Chinstrap', 'Dream'), ('Chinstrap', 'Torgersen'), ('Gentoo', 'Biscoe'), ('Gentoo', 'Dream'), ('Gentoo', 'Torgersen')])

In [19]:
# No ordering (timestamp_with_time after timestamp)
assert (dummy_df["timestamp_with_time"] > dummy_df["timestamp"]).all(), "timestamp_with_time is not always greater than timestamp"

AssertionError: timestamp_with_time is not always greater than timestamp

## Table Level DP with Keys and Dependencies

In [20]:
metadata = make_metadata_from_data(
    df, default_contributions_level="table_with_keys", privacy_unit="penguin_id", with_dependencies=True
)
with open("metadata/penguin_metadata_table_with_keys_with_dependencies.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [25]:
metadata

{'@context': ['http://www.w3.org/ns/csvw',
  '/home/onyxia/work/csvw-safe/csvw-safe-library/csvw-safe-context.jsonld'],
 '@type': 'Table',
 'privacyUnit': 'penguin_id',
 'maxContributions': 3,
 'maxLength': 344,
 'publicLength': 344,
 'tableSchema': {'columns': [{'@type': 'Column',
    'name': 'species',
    'datatype': <DataTypes.STRING: 'string'>,
    'required': True,
    'privacyId': False,
    'nullableProportion': 0.0,
    'rowDependencies': [{'dependsOn': 'island',
      'dependencyType': <DependencyType.MAPPING: 'mapping'>,
      'valueMap': {'Biscoe': ['Adelie', 'Gentoo'],
       'Dream': ['Adelie', 'Chinstrap'],
       'Torgersen': ['Adelie']}},
     {'dependsOn': 'penguin_id',
      'dependencyType': <DependencyType.FIXED: 'fixedPerEntity'>},
     {'dependsOn': 'timestamp_with_time',
      'dependencyType': <DependencyType.FIXED: 'fixedPerEntity'>}],
    'keyValues': ['Adelie', 'Chinstrap', 'Gentoo'],
    'invariantPublicKeys': True,
    'exhaustiveKeys': True,
    'maxNumPa

In [21]:
validated_metadata = validate_metadata(metadata)

In [22]:
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(5)

,species,penguin_id,timestamp,timestamp_with_time,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,favourite_number
0,Chinstrap,214,2025-04-24,2025-05-31 11:34:28,Torgersen,55.304240,NaN,<NA>,3469,False,0
1,Chinstrap,41,2025-12-10,2025-12-30 19:27:11,Dream,53.804308,16.988090,231,4015,False,0
2,Chinstrap,109,2025-04-30,2025-05-10 16:00:16,Torgersen,59.435736,18.055669,<NA>,3366,False,0
3,Gentoo,342,2025-03-24,2025-06-01 15:23:02,Biscoe,54.813824,21.422256,198,3867,False,0
4,Gentoo,320,2025-08-17,2025-10-22 07:46:22,Torgersen,41.119513,17.601551,<NA>,4544,False,0


In [23]:
# Only relevant combination
dummy_df.groupby(["species", "island"]).groups.keys()

dict_keys([('Adelie', 'Biscoe'), ('Adelie', 'Dream'), ('Adelie', 'Torgersen'), ('Chinstrap', 'Biscoe'), ('Chinstrap', 'Dream'), ('Chinstrap', 'Torgersen'), ('Gentoo', 'Biscoe'), ('Gentoo', 'Dream'), ('Gentoo', 'Torgersen')])

In [24]:
# No ordering (timestamp_with_time after timestamp)
assert (dummy_df["timestamp_with_time"] >= dummy_df["timestamp"]).all(), "timestamp_with_time is not always greater than timestamp"

## Column Level DP

In [24]:
metadata = make_metadata_from_data(
    df, privacy_unit="penguin_id", default_contributions_level="column", with_dependencies=False
)
with open("metadata/penguin_metadata_column_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [25]:
#metadata

In [26]:
validated_metadata = validate_metadata(metadata)

In [27]:
# Looks similar but has DP information
dummy_df = make_dummy_from_metadata(metadata, nb_rows=100, seed=0)
dummy_df.head(3)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,penguin_id,timestamp,timestamp_with_time,favourite_number
0,Chinstrap,Dream,NaN,17.271202,<NA>,5949,False,65,2025-03-20,2025-12-19 06:47:46,0
1,Chinstrap,Biscoe,42.977825,15.568964,229,4120,True,340,2025-05-14,2025-02-12 06:47:46,4
2,Chinstrap,Biscoe,54.151716,18.749224,211,2944,True,269,2025-03-10,2025-07-22 06:47:46,2


## Partition Level DP

In [28]:
metadata = make_metadata_from_data(
    df, privacy_unit="penguin_id", default_contributions_level="partition", with_dependencies=False
)
with open("metadata/penguin_metadata_partition_level.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [29]:
#metadata

## Partition Level DP with Continuous Partitions

In [30]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ["2025-01-02 00:00:00", "2025-06-02 00:00:00", "2025-12-30 00:00:00"],
}

In [31]:
metadata = make_metadata_from_data(
    df, 
    privacy_unit="penguin_id", 
    default_contributions_level="partition", 
    continuous_partitions=continuous_partitions, 
    with_dependencies=False
)
with open("metadata/penguin_metadata_partition_level_with_continuous.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [32]:
#metadata

## Per-Column DP Levels

In [33]:
continuous_partitions = {"bill_length_mm": [30.0, 40.0, 50.0, 60.0]}
fine_contrib = {
    "species": "partition",
    "island": "column"
    # "bill_length_mm" is implicitely partition
}

In [34]:
metadata = make_metadata_from_data(
    df, 
    privacy_unit="penguin_id", 
    default_contributions_level="table_with_keys",
    fine_contributions_level=fine_contrib,
    continuous_partitions=continuous_partitions,
    with_dependencies=False
)
with open("metadata/penguin_metadata_fine_contrib_levels.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [35]:
#metadata

## Column Level for Column Groups

In [36]:
column_groups = [
    ["species", "island"],
    ["species", "island", "favourite_number"]
]

In [37]:
metadata = make_metadata_from_data(
    df,
    privacy_unit="penguin_id",
    default_contributions_level="column",
    column_groups=column_groups,
    with_dependencies=False
)
with open("metadata/penguin_metadata_column_level_column_group.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [38]:
#metadata

## Partition Level for Column Groups

In [39]:
column_groups = [["species", "island"]]

In [40]:
metadata = make_metadata_from_data(
    df,
    privacy_unit="penguin_id",
    default_contributions_level="partition",
    column_groups=column_groups,
    with_dependencies=False
)
with open("metadata/penguin_metadata_partition_level_column_group.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [41]:
#metadata

## Partition Level for Column Groups with Continuous Partitions

In [42]:
continuous_partitions = {
    "bill_length_mm": [30.0, 40.0, 50.0, 60.0],
    "timestamp": ["2025-01-02 00:00:00", "2025-06-02 00:00:00", "2025-12-30 00:00:00"],
}
column_groups = [
    ["species", "bill_length_mm"],
]
fine_contrib = {
    "species": "partition",
    "island": "column"
    # "bill_length_mm" is implicitely partition
}

In [43]:
metadata = make_metadata_from_data(
    df,
    privacy_unit="penguin_id",
    default_contributions_level="table_with_keys",
    fine_contributions_level=fine_contrib,
    continuous_partitions=continuous_partitions,
    column_groups=column_groups,
)
with open("metadata/penguin_metadata_fine_levels_column_group_continuous.json-ld", "w", encoding="utf-8") as f:
    json.dump(metadata, f)

In [44]:
#metadata

... and any other combination.